In [1]:
import pandas as pd

In [2]:
#importar el dataset y generar un summary estadistico, ademas formatear la columnas de fechas
df = pd.read_csv('PdB_Dumps_Attrbs\ParsedObserverLogs_wDates.csv')

In [3]:
df.describe()

,Wind,Sea State,Ambient Noise,Water Depth,Swell Noise,Direccion
count,131.000000,131.000000,131.000000,131.000000,131.000000,129.000000
mean,15.778626,1.562341,8.484097,1262.672519,0.473282,177.906977
std,5.235295,0.427593,4.307204,179.200917,0.501202,90.326443
min,6.500000,0.500000,0.666667,0.000000,0.000000,90.000000
25%,12.500000,1.250000,5.000000,1235.425000,0.000000,90.000000
50%,15.000000,1.500000,7.500000,1268.650000,0.000000,90.000000
75%,17.500000,2.000000,11.000000,1345.500000,1.000000,270.000000
max,32.500000,3.250000,23.000000,1565.050000,1.000000,270.000000


Vamos a trabajar con la probabilidad marginal de Swell Noise, eso se calcula simplemente haciendo el conteo de todos los valores en el subset A y B, para luego normalizar sobre el numero de valores totales

In [9]:
A = df['Swell Noise'].sum()
B = (df['Swell Noise'] == 0).sum()

Prob_SwellNoise = A / (A + B)
Prob_NoSwellNoise = 1 - Prob_SwellNoise

print(f"Probabilidad de Swell Noise: {Prob_SwellNoise}")
print(f"Probabilidad de No Swell Noise: {Prob_NoSwellNoise}")

Probabilidad de Swell Noise: 0.4732824427480916
Probabilidad de No Swell Noise: 0.5267175572519084


In [4]:
#Formatear la columna de fechas, usa el argumento que asume el formato de fecha que se encuentra en la columna, en este caso es 'day/month/year'
df['FechaDeAdqui'] = pd.to_datetime(df['FechaDeAdqui'],dayfirst=True)

In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        'Distribucion de la marea',
        'Distribucion del ruido ambiental',
        'Distribucion del viento'
    )
)

SeaState_bins = 3
AmbientNoise_bins = 20
Wind_bins = 20

fig.add_trace(
    go.Histogram(
        x=df['Sea State'],
        name='Sea State',
        opacity=0.75,
        nbinsx=SeaState_bins,
        marker=dict(line=dict(width=2, color='black'))
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=df['Ambient Noise'],
        name='Ambient Noise',
        opacity=0.75,
        nbinsx=AmbientNoise_bins,
        marker=dict(line=dict(width=2, color='black'))
    ),
    row=1, col=2
)

fig.add_trace(
    go.Histogram(
        x=df['Wind'],
        name='Wind',
        opacity=0.75,
        nbinsx=Wind_bins,
        marker=dict(line=dict(width=2, color='black'))
    ),
    row=1, col=3
)

fig.update_layout(
    title_text='Distribucion de la marea, ruido ambiental y viento',
    showlegend=False
)

fig.update_xaxes(title_text='Marea [m]', row=1, col=1)
fig.update_xaxes(title_text='Ruido Ambiental [dB]', row=1, col=2)
fig.update_xaxes(title_text='Viento [knots]', row=1, col=3)

fig.update_yaxes(title_text='Conteo', row=1, col=1)
fig.update_yaxes(title_text='Conteo', row=1, col=2)
fig.update_yaxes(title_text='Conteo', row=1, col=3)

fig.show()

In [5]:
def crear_histograma(df, columna='Wind', cuantiles=4):
    # Calcular los cuantiles para la columna especificada
    quantile_values = df[columna].quantile([i / cuantiles for i in range(1, cuantiles)]).values

    # Crear etiquetas para los rangos de cuantiles
    labels = [f'Q{i + 1}' for i in range(cuantiles)]

    # Asignar cada valor a su correspondiente rango de cuantil
    df[f'{columna}_Quantile'] = pd.cut(df[columna], bins=[-float('inf')] + list(quantile_values) + [float('inf')], labels=labels)

    # Contar el número de registros en cada cuantil
    counts = df[f'{columna}_Quantile'].value_counts().sort_index()
    probabilidaes = df[f'{columna}_Quantile'].value_counts(normalize=True).sort_index()


    # Crear el histograma utilizando Plotly, sin compartir el mismo eje, para las probabilidades
    fig = make_subplots(
        rows=1,cols=1, 
        specs=[[{"secondary_y": True}]]
    )
    fig.add_trace(
        go.Bar(x=counts.index, y=counts.values, name='Conteo', marker=dict(line=dict(width=2, color='black'))),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=probabilidaes.index, y=probabilidaes.values, name='Probabilidad', mode='lines+markers', marker=dict(color='red')),
        secondary_y=True
    )
    fig.update_layout(
        title_text=f'Histograma de {columna} con cuantiles',
        xaxis_title=f'Cuantiles de {columna}',
        yaxis_title='Conteo',
        yaxis2_title='Probabilidad',
        showlegend=True
    )
    return fig

crear_histograma(df, columna='Wind', cuantiles=4).show()

In [5]:
#crear nuevo df para calcular y plotear las PMFs
import plotly.graph_objects as go
def plot_PMF(df, columna='Ambient Noise'):
    df_PMF = df.copy()
    df_PMF_columna = df_PMF[columna].value_counts(normalize=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df_PMF_columna.index, y=df_PMF_columna.values, name=f'PMF de {columna}', marker=dict(line=dict(width=2, color='black'))))
    fig.update_layout(
        title_text=f'PMF de {columna}',
        xaxis_title=f'{columna} [dB]',
        yaxis_title='Probabilidad',
        showlegend=True
    )
    return fig
plot_PMF(df, columna='Ambient Noise').show()

De la PMF de ambient noise podemos ver que el valor maximo que alcanza un valor es de 6%, esto sucede porque la suma total debe ser siempre 1 para que sea válida. 
Quiere decir que hay un 6% de probabilidad de obtener 4 o 4.5

Ahora vamos a trabajar con los distintos modelos probabilisticos;
1. Binomial
Acá el random variable son los exitos contados, es realizar varios ensayos de Bernoulli y contar si fue exito o fracaso. Tiene 2 parametros n - intentos y p - probabilidad.
2. Geometrico
3. Bernoulli
4. Uniforme

In [12]:
#Primero necesitamos un criterio de T o F, para ello vamos a usar la PMF

threshold = 15
p_ambientnoise = (df['Ambient Noise'] > threshold).mean()
print(f'Probabilidad de que el ruido ambiental sea mayor a {threshold} dB: {p_ambientnoise:.4f}')

Probabilidad de que el ruido ambiental sea mayor a 15 dB: 0.0840


In [ ]:
from scipy.stats import binom

n = 20

model = binom(n, p_ambientnoise)

In [17]:
import numpy as np
import plotly.graph_objects as go

k_values = np.arange(0, n + 1)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=k_values,
    y=model.pmf(k_values),
    name='Binomial PMF'
))
fig.update_layout(
    title=f'PMF Binomial(n={n}, p={p_ambientnoise:.4f})',
    xaxis_title='k (número de éxitos en 20 ensayos)',
    yaxis_title='P(X = k)'
)
fig.show()

# Probabilidad de observar más de 15 eventos ruidosos en 20 intentos
print(f"P(X > 15) = {model.sf(15):.4f}")
# Intervalo del 95%
print(f"Intervalo 95%: {model.interval(0.95)}")

P(X > 15) = 0.0000
Intervalo 95%: (np.float64(0.0), np.float64(4.0))


In [18]:
model.pmf(5)

np.float64(0.017366006654196534)

In [7]:
def plot_probabilidades_condicionales(df, columna, cuantiles):
    """
    Grafica P(Swell Noise = 0 | bin) y P(Swell Noise = 1 | bin)
    usando bins por cuantiles de la columna indicada.

    Parametros
    ----------
    df : pandas.DataFrame
        Dataframe de entrada.
    columna : str
        Variable condicionante para construir cuantiles.
    cuantiles : int
        Numero de cuantiles a usar en qcut.
    """
    df_condicional = df.copy()
    col_bin = f"{columna}_Binned"

    df_condicional[col_bin] = pd.qcut(
        df_condicional[columna],
        q=cuantiles,
        duplicates='drop'
    )

    probabilidades = (
        df_condicional
        .groupby(col_bin)['Swell Noise']
        .value_counts(normalize=True)
    )

    tabla_probabilidades = probabilidades.unstack().fillna(0)

    # Asegurar columnas 0 y 1 aunque alguna no aparezca en un bin.
    for clase in [0, 1]:
        if clase not in tabla_probabilidades.columns:
            tabla_probabilidades[clase] = 0.0

    tabla_probabilidades = tabla_probabilidades[[0, 1]]

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=tabla_probabilidades.index.astype(str),
            y=tabla_probabilidades[0],
            name='Swell Noise = 0',
            marker=dict(line=dict(width=2, color='black'))
        )
    )
    fig.add_trace(
        go.Bar(
            x=tabla_probabilidades.index.astype(str),
            y=tabla_probabilidades[1],
            name='Swell Noise = 1',
            marker=dict(line=dict(width=2, color='black'))
        )
    )

    fig.update_layout(
        title_text=f'Probabilidades condicionales de Swell Noise dado {columna}',
        xaxis_title=f'Cuantiles de {columna}',
        yaxis_title='Probabilidad',
        barmode='group',
        showlegend=True
    )
    fig.update_yaxes(range=[0, 1])

    return fig

plot_probabilidades_condicionales(df, columna='Wind', cuantiles=4).show()

In [8]:
plot_probabilidades_condicionales(df, columna='Ambient Noise', cuantiles=8).show()

In [110]:
# 1) Binning en 8 cuantiles
df_tb = df.copy()
df_tb["Ambient_Noise_Binned"] = pd.qcut(df_tb["Ambient Noise"], q=8, duplicates="drop")

# 2) Probabilidad condicional por cuantil
p_swell_por_bin = (
    df_tb.groupby("Ambient_Noise_Binned")["Swell Noise"]
    .mean()  # como Swell Noise es 0/1, mean = P(Swell=1 | bin)
    .rename("P_swell")
)

# 3) Asignar esa probabilidad a cada fila (traceback fila a fila)
df_tb = df_tb.join(p_swell_por_bin, on="Ambient_Noise_Binned")

# 4) Elegir criterio de riesgo (ejemplo: >= 0.6)
umbral = 0.2
traceback = df_tb.loc[df_tb["P_swell"] < umbral, [
    "Secuencia", "FechaDeAdqui", "Ambient Noise", "Ambient_Noise_Binned", "Swell Noise", "P_swell"
]].sort_values(["P_swell", "Ambient Noise"], ascending=[False, False])

traceback.head(20)

,Secuencia,FechaDeAdqui,Ambient Noise,Ambient_Noise_Binned,Swell Noise,P_swell
13,01165P063,08/08/2005,4.000000,"(0.666, 4.0]",0,0.157895
18,01275P028,31/07/2005,4.000000,"(0.666, 4.0]",0,0.157895
20,01319P024,30/07/2005,4.000000,"(0.666, 4.0]",0,0.157895
40,01517I059,08/08/2005,4.000000,"(0.666, 4.0]",1,0.157895
52,01649P039,02/08/2005,4.000000,"(0.666, 4.0]",1,0.157895
56,01737P031,31/07/2005,4.000000,"(0.666, 4.0]",0,0.157895
57,01759P029,31/07/2005,4.000000,"(0.666, 4.0]",0,0.157895
62,01847P019,28/07/2005,4.000000,"(0.666, 4.0]",0,0.157895
63,01869P017,28/07/2005,4.000000,"(0.666, 4.0]",0,0.157895
35,01495I064,09/08/2005,3.900000,"(0.666, 4.0]",0,0.157895


In [20]:
import duckdb
import re

path_noisy = r'PdB_Dumps_Attrbs/Attributes_byWindow_AmpFreq_RawSEGYInputShots_L02067I006'
path_clean = r"PdB_Dumps_Attrbs\Attributes_byWindow_AmpFreq_RawSEGYInputShots_L01759P029"
normalized_path_clean = "muestra_normalizada_clean.tsv"
normalized_path_noisy = "muestra_normalizada_noisy.tsv"
max_lines = 400000
sample_rows = 100000


def normalize_whitespace_to_tsv(input_path, output_path, max_lines=None):
    """Convierte separadores por espacios/tabs multiples a un TSV limpio."""
    with open(input_path, "r", encoding="utf-8", errors="replace") as fin, open(
        output_path, "w", encoding="utf-8"
    ) as fout:
        for i, line in enumerate(fin):
            if max_lines is not None and i >= max_lines:
                break
            clean = re.sub(r"[ \t]+", "\t", line.strip())
            if clean:
                fout.write(clean + "\n")


normalize_whitespace_to_tsv(path_clean, normalized_path_clean, max_lines=max_lines)
normalize_whitespace_to_tsv(path_noisy, normalized_path_noisy, max_lines=max_lines)

con = duckdb.connect()
tab_reader = "read_csv_auto(?, delim='\\t', header=true, sample_size=200000, ignore_errors=true)"

muestra_clean = con.execute(
    f"""
    SELECT *
    FROM {tab_reader}
    USING SAMPLE {sample_rows} ROWS
    """,
    [normalized_path_clean],
).df()
muestra_noisy = con.execute(
    f"""
    SELECT *
    FROM {tab_reader}
    USING SAMPLE {sample_rows} ROWS
    """,
    [normalized_path_noisy],
).df()

print(f"Muestra clean: {muestra_clean.shape}")
muestra_clean.head()

Muestra clean: (100000, 50)


,XCORD_SOURCE,YCORD_SOURCE,SAIL_SEQ_NUM,SHOTPOINT_NUM,FLD_CABLE_NUM,SAIL_LINE_NUM,TRACE_NUM,SOURCE_DETECT_DIST,TR.DOMFREQ_W0,TR.DOMFREQ_W1,...,TR.PSD66_125_W2,TR.PSD66_125_W3,TR.PSD66_125_W4,TR.RMSAMP_W0,TR.RMSAMP_W1,TR.RMSAMP_W2,TR.RMSAMP_W3,TR.RMSAMP_W4,NUM_SAMPS_NE_ZERO,NUM_SAMPS_EQ_ZERO
0,404434.0,7654584.0,29,2141,11,1759,350,4511.09,48.767963,45.527153,...,15.073156,1.256393,0.004789,0.147283,10.965183,6.675982,2.252102,0.195784,1642,23
1,404038.0,7654558.0,29,2120,9,1759,44,715.10,35.051544,57.108624,...,26.980366,8.701350,0.005085,0.284884,26.969019,10.468072,5.381004,0.284434,1642,23
2,403925.0,7654556.0,29,2114,2,1759,55,876.89,35.051544,56.309898,...,12.812403,15.581378,0.005305,0.281589,25.652683,10.776863,6.360268,0.285527,1642,23
3,404562.0,7654557.0,29,2148,9,1759,278,3621.49,42.094456,50.718845,...,9.480058,6.018655,0.003067,0.145072,9.711170,7.048928,5.943326,0.231365,1642,23
4,403871.0,7654582.0,29,2111,6,1759,57,862.01,40.979382,59.504787,...,19.478199,13.383112,0.005397,0.255076,24.302069,9.301752,5.078825,0.256221,1642,23


In [21]:
print(f"Muestra noisy: {muestra_noisy.shape}")
muestra_noisy.head()

Muestra noisy: (100000, 50)


,XCORD_SOURCE,YCORD_SOURCE,SAIL_SEQ_NUM,SHOTPOINT_NUM,FLD_CABLE_NUM,SAIL_LINE_NUM,TRACE_NUM,SOURCE_DETECT_DIST,TR.DOMFREQ_W0,TR.DOMFREQ_W1,...,TR.PSD66_125_W2,TR.PSD66_125_W3,TR.PSD66_125_W4,TR.RMSAMP_W0,TR.RMSAMP_W1,TR.RMSAMP_W2,TR.RMSAMP_W3,TR.RMSAMP_W4,NUM_SAMPS_NE_ZERO,NUM_SAMPS_EQ_ZERO
0,384268.0,7650732.0,6,1057,10,2067,184,2453.83,11.430678,49.920124,...,6.384092,2.943701,0.001332,0.141785,13.055648,5.345701,3.351463,0.159762,1642,23
1,384510.0,7650757.0,6,1070,5,2067,302,3919.37,3.242075,18.370605,...,2.350182,1.459913,0.009905,1.665761,15.922886,3.967614,2.449451,1.996210,1642,23
2,383422.0,7650753.0,6,1012,3,2067,162,2188.42,3.067485,43.530350,...,8.343184,8.201171,0.001775,0.574697,13.924324,5.302731,3.580024,0.584869,1642,23
3,383369.0,7650726.0,6,1009,5,2067,338,4365.92,9.585890,25.958464,...,4.796039,1.406182,0.001889,0.165323,14.832634,4.940214,3.052188,0.299618,1642,23
4,384004.0,7650738.0,6,1043,4,2067,252,3305.11,3.742515,37.539932,...,3.862509,5.329250,0.004035,1.233038,13.579308,11.604740,16.089273,2.225287,1642,23


In [103]:
from plotly.subplots import make_subplots


def calc_avg_by_group(df, attribute):
    import plotly.express as px
    import plotly.graph_objects as go
    
    # Group by the three variables and calculate mean
    df_grouped = df.groupby(['SHOTPOINT_NUM', 'FLD_CABLE_NUM', 'TRACE_NUM'], as_index=False)[attribute].mean()
    
    # Create subplots for each group variable
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(f'Mean {attribute} by SHOTPOINT_NUM', 
                       f'Mean {attribute} by FLD_CABLE_NUM',
                       f'Mean {attribute} by TRACE_NUM')
    )
    
    # Plot 1: by SHOTPOINT_NUM
    sp_avg = df_grouped.groupby('SHOTPOINT_NUM')[attribute].mean()
    fig.add_trace(
        go.Scatter(x=sp_avg.index, y=sp_avg.values, mode='lines+markers', name='SHOTPOINT_NUM'),
        row=1, col=1
    )
    
    # Plot 2: by FLD_CABLE_NUM
    fc_avg = df_grouped.groupby('FLD_CABLE_NUM')[attribute].mean()
    fig.add_trace(
        go.Scatter(x=fc_avg.index, y=fc_avg.values, mode='lines+markers', name='FLD_CABLE_NUM'),
        row=2, col=1
    )
    
    # Plot 3: by TRACE_NUM
    tn_avg = df_grouped.groupby('TRACE_NUM')[attribute].mean()
    fig.add_trace(
        go.Scatter(x=tn_avg.index, y=tn_avg.values, mode='lines+markers', name='TRACE_NUM'),
        row=3, col=1
    )
    
    fig.update_yaxes(title_text=f'Mean {attribute}', row=1, col=1)
    fig.update_yaxes(title_text=f'Mean {attribute}', row=2, col=1)
    fig.update_yaxes(title_text=f'Mean {attribute}', row=3, col=1)
    fig.update_xaxes(title_text='SHOTPOINT_NUM', row=1, col=1)
    fig.update_xaxes(title_text='FLD_CABLE_NUM', row=2, col=1)
    fig.update_xaxes(title_text='TRACE_NUM', row=3, col=1)
    
    fig.update_layout(height=900, showlegend=False, title_text=f'Analysis of {attribute}')
    return fig

calc_avg_by_group(muestra_noisy, attribute='TR.PSD0_10_W4')

In [97]:
def plot_PMF(df, attribute):
    import plotly.graph_objects as go
    pmf = df[attribute].value_counts(normalize=True)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=pmf.index,
        y=pmf.values,
        mode='markers',
        name=f'PMF de {attribute}',
        marker=dict(color='blue', line=dict(width=2, color='black'))
    )
    )

    for atrib,prob in pmf.items():
        fig.add_trace(go.Scatter(
            x=[atrib,atrib],
            y=[0,prob],
            mode='lines',
            line=dict(color='red', width=2),
            showlegend=False
        ))

    fig.update_layout(
        title_text=f'PMF de {attribute}',
        xaxis_title=attribute,
        yaxis_title='Probabilidad',
        showlegend=True
    )
    return fig
plot_PMF(muestra_noisy, attribute='TR.DOMFREQ_W0')

In [101]:
import numpy as np
from sklearn.neighbors import KernelDensity

def plot_PDF(df,attribute,bandwith=0.2,n_points=500):
    x = df[attribute].dropna().to_numpy().reshape(-1, 1)
    kde = KernelDensity(kernel='gaussian', bandwidth=bandwith).fit(x)

    x_d = np.linspace(x.min(), x.max(), n_points).reshape(-1, 1)
    log_density = kde.score_samples(x_d)
    pdf = np.exp(log_density)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_d.ravel(),
        y=pdf,
        mode='lines',
        name=f'PDF de {attribute}',
    ))
    fig.update_layout(
        title= f'PDF de {attribute} con KDE (bandwidth={bandwith})',
        xaxis_title=attribute,
        yaxis_title='Densidad',
        showlegend=True
    )
    return fig
plot_PDF(muestra_noisy, attribute='TR.PSD11_65_W1', bandwith=0.2)